<a href="https://colab.research.google.com/github/Marlon-Sbardelatti/machine-learning/blob/feature%2Fseminario/seminario/urban_fire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Instala dependências
!pip install datasets tensorflow scikit-learn pillow

In [8]:
from datasets import load_dataset

# Carrega dataset do HuggingFace
dataset = load_dataset(
    path='incrisvel/urban-fire-identification'
)

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

In [9]:
# Separar deterministicamente amostras de treino e teste
split_dataset = dataset["train"].train_test_split(
    test_size=0.5,
    seed=1
)

train_data = split_dataset["train"]
test_data = split_dataset["test"]

# Mostrar estrutura do dataset
split_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 402
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 402
    })
})

In [10]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model


# Cria modelo pré-treinado para classificação
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(224, 224, 3)
)

# Extração de features
def extract_features(dataset):
    images = []
    labels = []

    for sample in dataset:
        image = sample["image"]

        # Converte para RGB por garantia
        image = image.convert("RGB")

        # Redimensiona imagens para mesmo tamanho (padrão ImageNet)
        image = image.resize((224, 224))

        # Converte imagem para um ndarray
        image = img_to_array(image)

        images.append(image)
        labels.append(sample["label"])

    images = np.array(images)

    # Preprocessamento específico MobileNet
    images = preprocess_input(images)

    # Extrai embeddings
    features = base_model.predict(
        images,
        verbose=0
    )

    return features, np.array(labels)

In [11]:
# Extração de embeddings para treino e teste
X_train, y_train = extract_features(train_data)
X_test, y_test = extract_features(test_data)

In [12]:
from sklearn.preprocessing import StandardScaler

# Normalização dos embeddings para SVM
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
from sklearn.svm import SVC

# Treinamento do modelo SVM
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale'
)

svm.fit(X_train, y_train)

SVC()

In [14]:
from sklearn.metrics import classification_report, accuracy_score

# Predição
y_pred = svm.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))

print("\nRelatório da classificação:\n")
print(classification_report(y_test, y_pred))

Acurácia: 0.9751243781094527

Relatório da classificação:

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       200
           1       0.98      0.97      0.97       202

    accuracy                           0.98       402
   macro avg       0.98      0.98      0.98       402
weighted avg       0.98      0.98      0.98       402

